In [0]:
%pip install \
    azure-keyvault-secrets==4.7.0 \
    azure-identity==1.15.0 \
    azure-core==1.29.5 \
    azure-storage-file-datalake==12.14.0 \
    sseclient-py \
    openai \
    dotenv \
    confluent-kafka \
    great-expectations \
    altair==4.2.2 \
    redis

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC ## PULSE — Wikipedia EventStreams 수신 + AI 규칙 생성
# MAGIC - Wikipedia SSE → Bronze 적재
# MAGIC - AI 규칙 생성 (최초 1회)
# MAGIC - GX 품질 검사

# COMMAND ----------
import os
import sys
import json
import requests
from sseclient import SSEClient
from datetime import datetime

# ── 환경 설정 ────────────────────────────────────────────
os.environ["KEY_VAULT_URL"] = "https://kv-sense-team4.vault.azure.net/"
sys.path.insert(0, "/Workspace/Repos/3dt030@msacademy.msai.kr/3dt-3nd-project/src")

# ── vault 초기화 ─────────────────────────────────────────
import utils.vault_manager
utils.vault_manager._instance = None
from utils.vault_manager import get_vault_manager

vault = get_vault_manager()

# ── 연결 및 시크릿 로드 ──────────────────────────────────
storage_client    = vault.get_storage_client("datacopsadls")
kafka_producer    = vault.get_kafka_producer()

gx_openai_key        = vault.get_secret("gx-rulegen-openai-key")
gx_openai_endpoint   = vault.get_secret("gx-rulegen-openai-endpoint")
gx_openai_deployment = vault.get_secret("gx-rulegen-deployment-gpt-4-1-mini")
gx_openai_api_version = "2024-12-01-preview"

# ── 확인 ─────────────────────────────────────────────────
print("[OK] Key Vault 연결 완료")
print("[OK] ADLS 연결 완료")
print("[OK] Kafka Producer 연결 완료")
print("[OK] GX RuleGen OpenAI 설정 로드 완료")
print(f"[INFO] deployment = {gx_openai_deployment}")

In [0]:
# COMMAND ----------
from redis.cluster import RedisCluster, ClusterNode

redis_host = vault.get_secret("redis-host")
redis_password = vault.get_secret("redis-password")
redis_port = int(vault.get_secret("redis-port"))

redis_client = RedisCluster(
    startup_nodes=[
        ClusterNode(redis_host, redis_port)
    ],
    password=redis_password,
    ssl=True,
    ssl_check_hostname=False,
    decode_responses=True,
    skip_full_coverage_check=True,
    socket_connect_timeout=10,
    socket_timeout=10,
)

redis_client.ping()
print(f"[OK] Redis 재연결 완료: {redis_host}:{redis_port}")

# COMMAND ----------
deleted_rules = redis_client.delete(CACHE_KEY_RULES)
deleted_schema_hash = redis_client.delete(CACHE_KEY_SCHEMA + ":hash")
deleted_schema_detail = redis_client.delete(CACHE_KEY_SCHEMA + ":detail")

print("[OK] Redis 캐시 삭제 완료")
print(f"  rules 삭제 여부        : {deleted_rules}")
print(f"  schema hash 삭제 여부  : {deleted_schema_hash}")
print(f"  schema detail 삭제 여부: {deleted_schema_detail}")


# COMMAND ----------
print("rules:", redis_client.get(CACHE_KEY_RULES))
print("schema hash:", redis_client.get(CACHE_KEY_SCHEMA + ":hash"))
print("schema detail:", redis_client.get(CACHE_KEY_SCHEMA + ":detail"))